# EP19. 에이전트 가드레일 고급편

## 새벽 2시, 에이전트가 $3,000을 태웠습니다

---

### 학습 목표
1. **토큰 예산 미들웨어**를 구현하여 LLM 호출 비용을 세션 단위로 통제할 수 있다
2. **시간 제한 + 반복 제한** 가드레일로 에이전트의 무한 루프를 자동 종료할 수 있다
3. **LLM-as-Judge 품질 게이트**와 **LangGraph interrupt**를 결합하여 안전한 에이전트를 구축할 수 있다
4. **Graceful Degradation**(모델 다운그레이드)과 **Langfuse 모니터링**으로 프로덕션 안전장치를 완성할 수 있다

---

### AI 가이드
이 노트북은 **프로덕션 환경**에서 에이전트를 안전하게 운영하기 위한 가드레일 패턴을 다룹니다.
단순한 Rate Limiting이 아닌, **예산+시간+품질 삼중 제어**와 **Graceful Degradation**을 구현합니다.

### 사전 요구사항
- **EP06 디버깅**: LangSmith/Langfuse 기본 사용법
- **EP15 비용 최적화**: 토큰 비용 계산 기초

### 소요 시간: 약 70분

### 필요 API 키
- `ANTHROPIC_API_KEY` — Claude API
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` — Langfuse (선택)

---
## Section 1: 환경 설정

In [ ]:
# 필수 패키지 설치
!uv pip install langgraph langchain langchain-anthropic langfuse tiktoken python-dotenv --quiet

---
## Section 2: 라이브러리 로드 + API 키 + Langfuse 초기화

In [ ]:
import os
import json
import time
import asyncio
import logging
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, Literal

import tiktoken
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 환경 변수 로드
load_dotenv()

# 로깅 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("guardrails")

print("라이브러리 로드 완료")

In [ ]:
# API 키 확인
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요."
print("ANTHROPIC_API_KEY: 설정됨")

# Langfuse 초기화 (선택 — 키가 없으면 Mock 모드)
LANGFUSE_AVAILABLE = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))

if LANGFUSE_AVAILABLE:
    from langfuse import Langfuse
    langfuse = Langfuse()
    print("Langfuse: 연결됨")
else:
    # Mock Langfuse — 키 없이도 노트북 실행 가능
    class MockLangfuse:
        def trace(self, **kwargs): return self
        def event(self, **kwargs): pass
        def score(self, **kwargs): pass
        def generation(self, **kwargs): return self
        def end(self, **kwargs): pass
        @property
        def id(self): return "mock-trace-id"
    langfuse = MockLangfuse()
    print("Langfuse: Mock 모드 (키 미설정)")

In [ ]:
# LLM 인스턴스 생성 (기본: Sonnet)
llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    temperature=0,
    max_tokens=1024,
)

# 토크나이저 (토큰 수 추정용)
encoder = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """텍스트의 토큰 수를 추정합니다."""
    return len(encoder.encode(text))

# 테스트
test_text = "에이전트 가드레일은 프로덕션 환경에서 필수입니다."
print(f"테스트 텍스트 토큰 수: {count_tokens(test_text)}")

---
## Section 3: 토큰 예산 미들웨어

**핵심 아이디어**: 세션별로 토큰 예산을 부여하고, Hard Limit과 Soft Limit을 이중으로 관리합니다.

| 상태 | 예산 사용률 | 동작 |
|------|-----------|------|
| Green | 0-50% | 정상 운영, 모든 모델 사용 가능 |
| Yellow | 50-80% | 경고 로깅, Langfuse 이벤트 |
| Red | 80-100% | 모델 다운그레이드 시작 |
| Halted | 100%+ | 실행 거부, 인간 승인 필요 |

In [ ]:
@dataclass
class BudgetTracker:
    """세션별 토큰 예산을 추적하고 관리하는 클래스"""
    hard_limit: int = 50_000        # 절대 초과 불가 (토큰)
    soft_limit: int = 40_000        # 경고 임계값 (토큰)
    used_tokens: int = 0
    session_id: str = ""
    history: list = field(default_factory=list)

    @property
    def remaining(self) -> int:
        return max(0, self.hard_limit - self.used_tokens)

    @property
    def usage_pct(self) -> float:
        return self.used_tokens / self.hard_limit

    @property
    def status(self) -> str:
        pct = self.usage_pct
        if pct >= 1.0:
            return "HALTED"
        elif pct >= 0.8:
            return "RED"
        elif pct >= 0.5:
            return "YELLOW"
        return "GREEN"

    def check(self, requested_tokens: int) -> str:
        """요청된 토큰 수에 대한 예산 상태를 확인합니다."""
        if self.used_tokens + requested_tokens > self.hard_limit:
            return "HALT"
        elif self.used_tokens > self.soft_limit:
            return "DEGRADE"
        return "OK"

    def consume(self, tokens: int, description: str = ""):
        """토큰을 소비하고 이력을 기록합니다."""
        self.used_tokens += tokens
        self.history.append({
            "tokens": tokens,
            "cumulative": self.used_tokens,
            "status": self.status,
            "description": description,
            "timestamp": datetime.now().isoformat(),
        })
        logger.info(f"[Budget] {description}: +{tokens} tokens | 누적: {self.used_tokens}/{self.hard_limit} ({self.status})")

    def report(self) -> str:
        """예산 사용 리포트를 반환합니다."""
        return (
            f"예산 리포트\n"
            f"  상태: {self.status}\n"
            f"  사용: {self.used_tokens:,} / {self.hard_limit:,} 토큰\n"
            f"  잔여: {self.remaining:,} 토큰\n"
            f"  사용률: {self.usage_pct:.1%}\n"
            f"  호출 횟수: {len(self.history)}"
        )


# 테스트
budget = BudgetTracker(hard_limit=10_000, soft_limit=8_000, session_id="test-001")
print(f"초기 상태: {budget.status}")

budget.consume(3_000, "첫 번째 호출")
print(f"3K 후 상태: {budget.status}")

budget.consume(3_000, "두 번째 호출")
print(f"6K 후 상태: {budget.status}")

budget.consume(3_000, "세 번째 호출")
print(f"9K 후 상태: {budget.status}")

print(f"\n1K 추가 요청 가능? → {budget.check(1_000)}")
print(f"2K 추가 요청 가능? → {budget.check(2_000)}")
print(f"\n{budget.report()}")

---
## Section 4: 시간 예산 타임아웃

**두 가지 타임아웃**:
- **Wall Clock Timeout**: 전체 경과 시간 제한 (사용자 체감 시간)
- **Per-Step Timeout**: 개별 LLM 호출 타임아웃

In [ ]:
import functools


def timeout_decorator(timeout_sec: float):
    """동기 함수에 타임아웃을 적용하는 데코레이터"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            import signal

            def handler(signum, frame):
                raise TimeoutError(f"함수 '{func.__name__}'가 {timeout_sec}초를 초과했습니다.")

            old_handler = signal.signal(signal.SIGALRM, handler)
            signal.alarm(int(timeout_sec))
            try:
                result = func(*args, **kwargs)
            finally:
                signal.alarm(0)
                signal.signal(signal.SIGALRM, old_handler)
            return result
        return wrapper
    return decorator


async def run_with_timeout(coro, timeout_sec: float = 30.0):
    """비동기 코루틴에 타임아웃을 적용합니다."""
    try:
        result = await asyncio.wait_for(coro, timeout=timeout_sec)
        return {"status": "completed", "result": result}
    except asyncio.TimeoutError:
        logger.warning(f"타임아웃 발생: {timeout_sec}초 초과")
        return {"status": "timeout", "action": "escalate", "timeout_sec": timeout_sec}


@dataclass
class TimeBudget:
    """시간 예산을 관리하는 클래스"""
    wall_timeout: float = 120.0    # 전체 타임아웃 (초)
    step_timeout: float = 30.0     # 개별 스텝 타임아웃 (초)
    start_time: float = field(default_factory=time.time)

    @property
    def elapsed(self) -> float:
        return time.time() - self.start_time

    @property
    def remaining(self) -> float:
        return max(0.0, self.wall_timeout - self.elapsed)

    @property
    def is_expired(self) -> bool:
        return self.elapsed >= self.wall_timeout

    def effective_step_timeout(self) -> float:
        """남은 시간과 스텝 타임아웃 중 작은 값을 반환합니다."""
        return min(self.step_timeout, self.remaining)


# 테스트: 비동기 타임아웃
async def slow_task():
    """의도적으로 느린 태스크"""
    await asyncio.sleep(5)
    return "완료"

async def fast_task():
    """빠른 태스크"""
    await asyncio.sleep(0.1)
    return "빠르게 완료"

# 테스트 실행
result_fast = await run_with_timeout(fast_task(), timeout_sec=2.0)
print(f"빠른 태스크: {result_fast}")

result_slow = await run_with_timeout(slow_task(), timeout_sec=1.0)
print(f"느린 태스크: {result_slow}")

# TimeBudget 테스트
timer = TimeBudget(wall_timeout=60.0, step_timeout=10.0)
print(f"\n경과: {timer.elapsed:.1f}초 | 잔여: {timer.remaining:.1f}초 | 만료: {timer.is_expired}")

---
## Section 5: 재귀 루프 카운터

에이전트가 "결과 불만족 → 재시도" 루프에 빠지면 비용과 시간이 폭발합니다.
`IterationLimiter`로 최대 반복 횟수를 강제합니다.

In [ ]:
@dataclass
class IterationLimiter:
    """에이전트 반복 횟수를 제한하는 클래스"""
    max_iterations: int = 10
    current: int = 0
    history: list = field(default_factory=list)

    def step(self, description: str = "") -> bool:
        """한 스텝을 진행합니다. True면 계속, False면 중단."""
        self.current += 1
        self.history.append({
            "iteration": self.current,
            "description": description,
            "timestamp": datetime.now().isoformat(),
        })

        if self.current >= self.max_iterations:
            logger.warning(
                f"반복 한계 도달: {self.current}/{self.max_iterations} — 자동 종료"
            )
            return False
        return True

    @property
    def remaining(self) -> int:
        return max(0, self.max_iterations - self.current)

    def reset(self):
        """카운터를 초기화합니다."""
        self.current = 0
        self.history.clear()


# 테스트: 최대 5회 반복
limiter = IterationLimiter(max_iterations=5)

for i in range(8):  # 의도적으로 한계 초과 시도
    can_continue = limiter.step(f"작업 {i+1}")
    print(f"  반복 {i+1}: continue={can_continue} | 잔여: {limiter.remaining}")
    if not can_continue:
        print("  루프 자동 종료!")
        break

print(f"\n총 반복: {limiter.current}/{limiter.max_iterations}")

---
## Section 6: LangGraph interrupt 사용법

**interrupt**는 LangGraph에서 그래프 실행을 일시 중단하고, 사용자 입력을 기다리는 메커니즘입니다.
위험한 작업(데이터 삭제, 대량 전송 등) 전에 인간 승인을 요청할 때 사용합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator


# 상태 정의
class AgentState(TypedDict):
    task: str
    risk_level: str
    approved: bool
    result: str
    messages: Annotated[list, operator.add]


def analyze_risk(state: AgentState) -> dict:
    """태스크의 위험도를 분석합니다."""
    task = state["task"].lower()
    dangerous_keywords = ["삭제", "delete", "drop", "전체", "all", "대량"]

    if any(kw in task for kw in dangerous_keywords):
        risk = "HIGH"
    else:
        risk = "LOW"

    logger.info(f"위험도 분석: '{task[:30]}...' → {risk}")
    return {"risk_level": risk, "messages": [f"위험도 분석 완료: {risk}"]}


def human_approval(state: AgentState) -> dict:
    """고위험 작업에 대해 인간 승인을 요청합니다."""
    if state["risk_level"] == "HIGH":
        # interrupt: 그래프 실행을 중단하고 사용자 입력 대기
        answer = interrupt(
            f"고위험 작업 감지!\n"
            f"작업: {state['task']}\n"
            f"승인하시겠습니까? (yes/no)"
        )
        approved = answer.lower() in ["yes", "y", "승인"]
        return {"approved": approved, "messages": [f"인간 승인: {'승인됨' if approved else '거부됨'}"]}

    return {"approved": True, "messages": ["저위험 — 자동 승인"]}


def execute_task(state: AgentState) -> dict:
    """승인된 작업을 실행합니다."""
    if not state.get("approved", False):
        return {"result": "작업이 거부되었습니다.", "messages": ["작업 거부됨"]}

    result = f"작업 '{state['task']}' 실행 완료 (시뮬레이션)"
    return {"result": result, "messages": [result]}


# 그래프 구성
builder = StateGraph(AgentState)
builder.add_node("analyze_risk", analyze_risk)
builder.add_node("human_approval", human_approval)
builder.add_node("execute_task", execute_task)

builder.add_edge(START, "analyze_risk")
builder.add_edge("analyze_risk", "human_approval")
builder.add_edge("human_approval", "execute_task")
builder.add_edge("execute_task", END)

# 체크포인터 (interrupt 상태 저장용)
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("LangGraph interrupt 그래프 구성 완료")

In [ ]:
# 테스트 1: 저위험 작업 (interrupt 없이 바로 실행)
config = {"configurable": {"thread_id": "safe-task-001"}}
result = graph.invoke(
    {"task": "고객 목록 조회", "approved": False, "result": "", "messages": []},
    config=config,
)
print(f"저위험 작업 결과: {result['result']}")
print(f"메시지 이력: {result['messages']}")

In [ ]:
# 테스트 2: 고위험 작업 (interrupt 발생 → 승인 → 실행)
config_danger = {"configurable": {"thread_id": "danger-task-001"}}

# 1단계: 그래프 실행 → interrupt에서 중단됨
result_1 = graph.invoke(
    {"task": "고객 데이터 1000건 삭제", "approved": False, "result": "", "messages": []},
    config=config_danger,
)
print("1단계: interrupt 발생 — 사용자 승인 대기 중")
print(f"  현재 상태: {result_1}")

# 2단계: 사용자 승인 전달 → 실행 재개
result_2 = graph.invoke(
    Command(resume="yes"),
    config=config_danger,
)
print(f"\n2단계: 승인 후 결과: {result_2['result']}")
print(f"  메시지 이력: {result_2['messages']}")

---
## Section 7: 품질 게이트 (LLM-as-Judge)

에이전트의 응답 품질을 LLM으로 평가하고, 기준 미달 시 자동으로 재시도하거나 에스컬레이션합니다.

In [ ]:
JUDGE_SYSTEM_PROMPT = """당신은 AI 응답 품질 평가 전문가입니다.
주어진 질문에 대한 응답의 품질을 1-10점으로 평가하세요.

평가 기준:
- 정확성: 사실적으로 올바른가?
- 완전성: 질문에 충분히 답했는가?
- 관련성: 질문과 관련된 내용인가?

반드시 아래 JSON 형식으로만 응답하세요:
{"score": <1-10>, "reason": "<한국어 평가 사유>"}"""


async def llm_judge(question: str, response: str) -> dict:
    """LLM-as-Judge로 응답 품질을 평가합니다."""
    judge_llm = ChatAnthropic(
        model="claude-sonnet-4-20250514",
        temperature=0,
        max_tokens=256,
    )

    judge_prompt = f"""[원래 질문]: {question}

[에이전트 응답]: {response}

위 응답의 품질을 JSON으로 평가하세요."""

    result = await judge_llm.ainvoke([
        SystemMessage(content=JUDGE_SYSTEM_PROMPT),
        HumanMessage(content=judge_prompt),
    ])

    try:
        parsed = json.loads(result.content)
        return parsed
    except json.JSONDecodeError:
        logger.warning(f"Judge 응답 파싱 실패: {result.content[:100]}")
        return {"score": 5, "reason": "파싱 실패 — 기본 점수"}


async def quality_gate(
    question: str,
    response: str,
    threshold: int = 7,
) -> dict:
    """품질 게이트: 점수에 따라 accept/retry/escalate를 결정합니다."""
    evaluation = await llm_judge(question, response)
    score = evaluation.get("score", 5)
    reason = evaluation.get("reason", "")

    if score >= threshold:
        action = "accept"
    elif score >= 4:
        action = "retry"
    else:
        action = "escalate"

    logger.info(f"[QualityGate] score={score}, action={action}, reason={reason[:50]}")
    return {"action": action, "score": score, "reason": reason}


print("품질 게이트 함수 정의 완료")

In [ ]:
# 품질 게이트 테스트
question = "Python에서 리스트와 튜플의 차이점은 무엇인가요?"

# 좋은 응답
good_response = (
    "리스트는 변경 가능(mutable)하고 대괄호[]를 사용하며, "
    "튜플은 변경 불가능(immutable)하고 소괄호()를 사용합니다. "
    "리스트는 요소 추가/삭제가 가능하지만, 튜플은 생성 후 수정할 수 없어 해시 가능하며 딕셔너리 키로 사용할 수 있습니다."
)

# 나쁜 응답
bad_response = "Python은 프로그래밍 언어입니다."

print("=== 좋은 응답 평가 ===")
result_good = await quality_gate(question, good_response)
print(f"결과: {result_good}")

print("\n=== 나쁜 응답 평가 ===")
result_bad = await quality_gate(question, bad_response)
print(f"결과: {result_bad}")

---
## Section 8: 삼중 가드레일 통합 에이전트

**예산 + 시간 + 품질**을 하나의 LangGraph 에이전트에 통합합니다.
이것이 프로덕션 에이전트의 핵심 안전장치입니다.

In [ ]:
@dataclass
class GuardrailResult:
    """가드레일 실행 결과를 담는 데이터 클래스"""
    status: Literal["ACCEPT", "HALT", "TIMEOUT", "ESCALATE", "ITERATION_LIMIT"]
    result: str
    metadata: dict = field(default_factory=dict)


class TripleGuardedAgent:
    """예산+시간+품질 삼중 가드레일이 적용된 에이전트"""

    def __init__(
        self,
        llm,
        budget: BudgetTracker,
        timer: TimeBudget,
        limiter: IterationLimiter,
        quality_threshold: int = 7,
        max_retries: int = 2,
    ):
        self.llm = llm
        self.budget = budget
        self.timer = timer
        self.limiter = limiter
        self.quality_threshold = quality_threshold
        self.max_retries = max_retries
        self.guardrail_events = []

    def _log_event(self, guardrail_type: str, action: str, details: str):
        event = {
            "type": guardrail_type,
            "action": action,
            "details": details,
            "timestamp": datetime.now().isoformat(),
            "budget_status": self.budget.status,
            "iteration": self.limiter.current,
        }
        self.guardrail_events.append(event)
        logger.info(f"[Guardrail] {guardrail_type}: {action} — {details}")

    async def _call_llm(self, question: str) -> str:
        """LLM 호출 (토큰 예산 차감 포함)"""
        input_tokens = count_tokens(question)
        estimated_total = input_tokens * 3  # 출력 포함 추정

        # 예산 확인
        budget_status = self.budget.check(estimated_total)
        if budget_status == "HALT":
            self._log_event("budget", "halt", f"예산 초과 (예상 {estimated_total} 토큰)")
            raise BudgetExhaustedError("토큰 예산 소진")

        # LLM 호출
        result = await self.llm.ainvoke([HumanMessage(content=question)])
        response_text = result.content

        # 실제 사용량 기록
        actual_tokens = count_tokens(question) + count_tokens(response_text)
        self.budget.consume(actual_tokens, f"LLM 호출 (반복 {self.limiter.current})")

        if budget_status == "DEGRADE":
            self._log_event("budget", "degrade", f"Soft Limit 초과 — 다운그레이드 권고")

        return response_text

    async def run(self, question: str) -> GuardrailResult:
        """삼중 가드레일이 적용된 에이전트 실행"""
        retries = 0

        while self.limiter.step(f"질문 처리 (재시도 {retries})"):
            # 1. 시간 확인
            if self.timer.is_expired:
                self._log_event("time", "timeout", f"Wall Clock 만료: {self.timer.elapsed:.1f}초")
                return GuardrailResult("TIMEOUT", "시간 예산 소진 — 실행 중단")

            # 2. LLM 호출 (예산 가드레일 내장)
            try:
                step_timeout = self.timer.effective_step_timeout()
                response = await asyncio.wait_for(
                    self._call_llm(question),
                    timeout=step_timeout,
                )
            except BudgetExhaustedError:
                return GuardrailResult("HALT", "토큰 예산 소진 — 실행 중단")
            except asyncio.TimeoutError:
                self._log_event("time", "step_timeout", f"스텝 타임아웃: {step_timeout:.1f}초")
                return GuardrailResult("TIMEOUT", "개별 스텝 시간 초과")

            # 3. 품질 평가
            quality = await quality_gate(question, response, self.quality_threshold)

            if quality["action"] == "accept":
                self._log_event("quality", "accept", f"score={quality['score']}")
                return GuardrailResult(
                    "ACCEPT", response,
                    metadata={"score": quality["score"], "retries": retries}
                )
            elif quality["action"] == "escalate":
                self._log_event("quality", "escalate", f"score={quality['score']} — 인간 개입 필요")
                return GuardrailResult("ESCALATE", f"품질 미달 (score={quality['score']})")
            else:
                # retry
                retries += 1
                self._log_event("quality", "retry", f"score={quality['score']}, retry={retries}")
                if retries >= self.max_retries:
                    self._log_event("quality", "max_retries", f"최대 재시도 도달: {retries}")
                    return GuardrailResult("ESCALATE", f"재시도 {retries}회 후에도 품질 미달")
                question = f"(이전 응답이 품질 미달이었습니다. 더 정확하고 완전하게 답변해주세요.)\n\n{question}"

        # 반복 한계 도달
        self._log_event("iteration", "limit", f"반복 한계: {self.limiter.current}")
        return GuardrailResult("ITERATION_LIMIT", "반복 한계 도달 — 마지막 결과 없음")


class BudgetExhaustedError(Exception):
    pass


print("TripleGuardedAgent 정의 완료")

In [ ]:
# 삼중 가드레일 에이전트 테스트
agent = TripleGuardedAgent(
    llm=llm,
    budget=BudgetTracker(hard_limit=50_000, soft_limit=40_000, session_id="triple-001"),
    timer=TimeBudget(wall_timeout=120.0, step_timeout=30.0),
    limiter=IterationLimiter(max_iterations=5),
    quality_threshold=7,
    max_retries=2,
)

result = await agent.run("Python에서 async/await의 동작 원리를 설명해주세요.")

print(f"\n최종 결과:")
print(f"  상태: {result.status}")
print(f"  메타데이터: {result.metadata}")
print(f"  응답: {result.result[:200]}...")
print(f"\n{agent.budget.report()}")
print(f"\n가드레일 이벤트: {len(agent.guardrail_events)}건")
for evt in agent.guardrail_events:
    print(f"  [{evt['type']}] {evt['action']}: {evt['details']}")

---
## Section 9: Graceful Degradation (모델 다운그레이드)

예산이 소진될수록 모델을 자동으로 다운그레이드하여 서비스 중단 없이 비용을 절감합니다.

**Frontier → Mid → Small → Refuse**

In [ ]:
MODEL_CHAIN = [
    {"name": "claude-opus-4-20250514",   "cost_per_1k_input": 0.015, "cost_per_1k_output": 0.075, "tier": "Frontier"},
    {"name": "claude-sonnet-4-20250514", "cost_per_1k_input": 0.003, "cost_per_1k_output": 0.015, "tier": "Mid"},
    {"name": "claude-haiku-3-20250307",  "cost_per_1k_input": 0.00025, "cost_per_1k_output": 0.00125, "tier": "Small"},
]


class GracefulDegradation:
    """예산 상태에 따라 모델을 자동으로 다운그레이드합니다."""

    def __init__(self, budget: BudgetTracker):
        self.budget = budget
        self.model_chain = MODEL_CHAIN
        self.downgrade_history = []

    def select_model(self) -> dict:
        """현재 예산 상태에 따라 적절한 모델을 선택합니다."""
        pct = self.budget.usage_pct

        if pct < 0.5:
            selected = self.model_chain[0]  # Opus
        elif pct < 0.8:
            selected = self.model_chain[1]  # Sonnet
        elif pct < 1.0:
            selected = self.model_chain[2]  # Haiku
        else:
            raise BudgetExhaustedError("예산 완전 소진 — 실행 거부")

        self.downgrade_history.append({
            "model": selected["name"],
            "tier": selected["tier"],
            "budget_pct": f"{pct:.1%}",
            "timestamp": datetime.now().isoformat(),
        })

        return selected

    async def invoke(self, question: str) -> dict:
        """자동 다운그레이드 적용 LLM 호출"""
        model_info = self.select_model()
        model_name = model_info["name"]

        logger.info(f"[Degradation] 선택 모델: {model_name} ({model_info['tier']}) | 예산: {self.budget.usage_pct:.1%}")

        llm_instance = ChatAnthropic(
            model=model_name,
            temperature=0,
            max_tokens=1024,
        )

        result = await llm_instance.ainvoke([HumanMessage(content=question)])

        # 토큰 사용량 기록
        tokens_used = count_tokens(question) + count_tokens(result.content)
        self.budget.consume(tokens_used, f"{model_info['tier']} 모델 호출")

        return {
            "model": model_name,
            "tier": model_info["tier"],
            "response": result.content,
            "tokens_used": tokens_used,
            "budget_status": self.budget.status,
        }


print("GracefulDegradation 정의 완료")
print(f"\n모델 체인:")
for m in MODEL_CHAIN:
    print(f"  {m['tier']:10s} | {m['name']:35s} | in=${m['cost_per_1k_input']}/1K, out=${m['cost_per_1k_output']}/1K")

In [ ]:
# Graceful Degradation 시뮬레이션
# 작은 예산으로 설정하여 다운그레이드 과정을 관찰합니다
small_budget = BudgetTracker(hard_limit=3_000, soft_limit=2_400, session_id="degrade-001")
degradation = GracefulDegradation(budget=small_budget)

test_questions = [
    "Python에서 GIL이란 무엇인가요? (한 문장으로)",
    "FastAPI의 장점을 한 줄로 설명해주세요.",
    "Docker란? (한 줄)",
]

for i, q in enumerate(test_questions):
    print(f"\n--- 호출 {i+1}: {q} ---")
    try:
        result = await degradation.invoke(q)
        print(f"  모델: {result['model']} ({result['tier']})")
        print(f"  토큰: {result['tokens_used']} | 예산: {small_budget.status}")
        print(f"  응답: {result['response'][:100]}...")
    except BudgetExhaustedError as e:
        print(f"  예산 소진! {e}")
        break

print(f"\n{small_budget.report()}")
print(f"\n다운그레이드 이력:")
for h in degradation.downgrade_history:
    print(f"  {h['tier']:10s} | 예산: {h['budget_pct']}")

---
## Section 10: Langfuse 가드레일 이벤트 로깅

가드레일 트리거 이벤트를 Langfuse에 기록하여 모니터링하고 알림을 설정합니다.

In [ ]:
class GuardrailLogger:
    """가드레일 이벤트를 Langfuse에 기록하는 로거"""

    def __init__(self, langfuse_client):
        self.client = langfuse_client
        self.events = []

    def start_trace(self, session_id: str, task: str) -> object:
        """새 트레이스를 시작합니다."""
        trace = self.client.trace(
            name="guarded_agent_run",
            session_id=session_id,
            metadata={"task": task[:100]},
        )
        return trace

    def log_guardrail_event(
        self,
        trace,
        guardrail_type: str,
        action: str,
        details: str,
    ):
        """가드레일 트리거 이벤트를 기록합니다."""
        event_data = {
            "type": guardrail_type,
            "action": action,
            "details": details,
            "timestamp": datetime.now().isoformat(),
        }
        self.events.append(event_data)

        # Langfuse에 이벤트 기록
        level = "WARNING" if action in ["warn", "degrade", "retry"] else "ERROR"
        self.client.event(
            trace_id=trace.id if hasattr(trace, 'id') else "mock",
            name=f"guardrail_{guardrail_type}",
            metadata=event_data,
            level=level,
        )

        logger.info(f"[Langfuse] {guardrail_type}/{action}: {details}")

    def log_budget_score(self, trace, budget: BudgetTracker):
        """예산 사용률을 Langfuse score로 기록합니다."""
        self.client.score(
            trace_id=trace.id if hasattr(trace, 'id') else "mock",
            name="budget_usage_pct",
            value=budget.usage_pct,
            comment=f"사용: {budget.used_tokens:,} / 한계: {budget.hard_limit:,}",
        )

    def log_quality_score(self, trace, score: int, action: str):
        """품질 점수를 Langfuse score로 기록합니다."""
        self.client.score(
            trace_id=trace.id if hasattr(trace, 'id') else "mock",
            name="quality_gate_score",
            value=score / 10.0,
            comment=f"score={score}, action={action}",
        )

    def summary(self) -> dict:
        """이벤트 요약을 반환합니다."""
        from collections import Counter
        type_counts = Counter(e["type"] for e in self.events)
        action_counts = Counter(e["action"] for e in self.events)
        return {
            "total_events": len(self.events),
            "by_type": dict(type_counts),
            "by_action": dict(action_counts),
        }


# 테스트
gl = GuardrailLogger(langfuse)
trace = gl.start_trace("test-session", "Langfuse 이벤트 로깅 테스트")

# 다양한 가드레일 이벤트 기록
gl.log_guardrail_event(trace, "budget", "warn", "Soft Limit 50% 도달")
gl.log_guardrail_event(trace, "budget", "degrade", "Soft Limit 80% — Sonnet으로 다운그레이드")
gl.log_guardrail_event(trace, "time", "timeout", "30초 스텝 타임아웃")
gl.log_guardrail_event(trace, "quality", "retry", "score=5 — 재시도")
gl.log_guardrail_event(trace, "iteration", "limit", "최대 반복 10회 도달")

print(f"이벤트 요약: {gl.summary()}")

---
## Section 11: 가드레일 트리거 테스트 시나리오

각 가드레일이 정확히 작동하는지 의도적으로 트리거하는 테스트 시나리오입니다.
프로덕션 배포 전 반드시 실행해야 합니다.

In [ ]:
# 테스트 시나리오 1: 토큰 예산 소진
print("=" * 60)
print("시나리오 1: 토큰 예산 Hard Limit 트리거")
print("=" * 60)

tiny_budget = BudgetTracker(hard_limit=500, soft_limit=400, session_id="test-budget")
tiny_agent = TripleGuardedAgent(
    llm=llm,
    budget=tiny_budget,
    timer=TimeBudget(wall_timeout=60.0, step_timeout=30.0),
    limiter=IterationLimiter(max_iterations=5),
    quality_threshold=7,
)

# 예산을 거의 다 소진시킨 상태에서 시작
tiny_budget.consume(490, "사전 소비 (시뮬레이션)")

result = await tiny_agent.run("Python의 역사에 대해 장문으로 설명해주세요.")
print(f"\n결과: {result.status}")
print(f"메시지: {result.result}")
print(f"\n가드레일 이벤트:")
for evt in tiny_agent.guardrail_events:
    print(f"  [{evt['type']}] {evt['action']}: {evt['details']}")

In [ ]:
# 테스트 시나리오 2: 시간 초과
print("=" * 60)
print("시나리오 2: Wall Clock Timeout 트리거")
print("=" * 60)

# 매우 짧은 타임아웃
expired_timer = TimeBudget(wall_timeout=0.001, step_timeout=0.001)
# 타이머가 즉시 만료되도록 잠시 대기
await asyncio.sleep(0.01)

timeout_agent = TripleGuardedAgent(
    llm=llm,
    budget=BudgetTracker(hard_limit=50_000, soft_limit=40_000, session_id="test-timeout"),
    timer=expired_timer,
    limiter=IterationLimiter(max_iterations=5),
    quality_threshold=7,
)

result = await timeout_agent.run("Hello")
print(f"\n결과: {result.status}")
print(f"메시지: {result.result}")
print(f"\n가드레일 이벤트:")
for evt in timeout_agent.guardrail_events:
    print(f"  [{evt['type']}] {evt['action']}: {evt['details']}")

In [ ]:
# 테스트 시나리오 3: 반복 한계
print("=" * 60)
print("시나리오 3: Iteration Limit 트리거")
print("=" * 60)

# max_iterations=1로 설정 → 품질이 미달이어도 재시도 불가
iter_agent = TripleGuardedAgent(
    llm=llm,
    budget=BudgetTracker(hard_limit=50_000, soft_limit=40_000, session_id="test-iter"),
    timer=TimeBudget(wall_timeout=120.0, step_timeout=30.0),
    limiter=IterationLimiter(max_iterations=1),
    quality_threshold=10,  # 거의 불가능한 기준
)

result = await iter_agent.run("1+1은?")
print(f"\n결과: {result.status}")
print(f"메시지: {result.result}")
print(f"\n가드레일 이벤트:")
for evt in iter_agent.guardrail_events:
    print(f"  [{evt['type']}] {evt['action']}: {evt['details']}")

In [ ]:
# 모든 테스트 결과 요약
print("=" * 60)
print("가드레일 트리거 테스트 요약")
print("=" * 60)

test_results = {
    "예산 소진 (HALT)": "HALT" in [e["action"] for e in tiny_agent.guardrail_events],
    "시간 초과 (TIMEOUT)": "timeout" in [e["action"] for e in timeout_agent.guardrail_events],
    "반복 한계 (ITERATION)": any(
        e["type"] in ["iteration", "quality"] for e in iter_agent.guardrail_events
    ),
}

for name, triggered in test_results.items():
    status = "PASS" if triggered else "FAIL"
    print(f"  [{status}] {name}")

all_passed = all(test_results.values())
print(f"\n전체 결과: {'모든 가드레일 정상 작동' if all_passed else '일부 가드레일 확인 필요'}")

---
## Section 12: Exercise

아래 두 Exercise를 완성하세요.

### Exercise 1: 커스텀 삼중 가드레일 에이전트

**목표**: 아래 요구사항에 맞는 삼중 가드레일 에이전트를 직접 구현하세요.

**요구사항**:
1. `BudgetTracker` — hard_limit=10,000, soft_limit=8,000
2. `TimeBudget` — wall_timeout=30초, step_timeout=15초
3. `quality_gate` — threshold=7, max_retries=2
4. 모든 가드레일 트리거 이벤트를 `GuardrailLogger`로 Langfuse에 기록
5. 3가지 테스트 시나리오 작성:
   - (a) 정상 실행 (모든 가드레일 통과)
   - (b) 예산 초과로 HALT
   - (c) 시간 초과로 TIMEOUT

In [ ]:
# TODO: Exercise 1 — 커스텀 삼중 가드레일 에이전트 구현

# Step 1: 가드레일 컴포넌트 초기화
# ex1_budget = BudgetTracker(hard_limit=???, soft_limit=???, session_id="ex1")
# ex1_timer = TimeBudget(wall_timeout=???, step_timeout=???)
# ex1_limiter = IterationLimiter(max_iterations=???)
# ex1_logger = GuardrailLogger(langfuse)

# Step 2: 에이전트 생성
# ex1_agent = TripleGuardedAgent(...)

# Step 3: 테스트 시나리오 (a) 정상 실행
# result_a = await ex1_agent.run("간단한 질문")

# Step 4: 테스트 시나리오 (b) 예산 초과
# 힌트: budget.consume()으로 예산을 미리 소진시키세요

# Step 5: 테스트 시나리오 (c) 시간 초과
# 힌트: wall_timeout을 매우 짧게 설정하세요

# Step 6: Langfuse 이벤트 기록 및 요약 출력

print("Exercise 1: TODO — 위 코드를 완성하세요")

### Exercise 2: Graceful Degradation 파이프라인

**목표**: 예산 소진에 따라 모델을 자동으로 다운그레이드하는 완전한 파이프라인을 구현하세요.

**요구사항**:
1. `GracefulDegradation` 클래스를 확장하여:
   - 각 모델 호출 시 비용을 달러로 추정
   - 총 비용 리포트 생성
2. 5개의 질문을 순차 처리하면서 다운그레이드 관찰
3. 각 호출의 모델, 토큰, 예상 비용을 테이블로 출력
4. Langfuse에 모델 전환 이벤트 기록
5. (보너스) 다운그레이드 시 사용자에게 알림 메시지 생성

In [ ]:
# TODO: Exercise 2 — Graceful Degradation 파이프라인 구현

# Step 1: GracefulDegradation 확장 (비용 추정 기능 추가)
# class CostAwareDegradation(GracefulDegradation):
#     def __init__(self, budget):
#         super().__init__(budget)
#         self.cost_history = []
#
#     def estimate_cost(self, model_info, tokens):
#         """토큰 수와 모델 가격으로 비용을 추정합니다."""
#         pass  # TODO
#
#     async def invoke_with_cost(self, question):
#         """비용 추적이 포함된 LLM 호출"""
#         pass  # TODO

# Step 2: 테스트 질문 5개 정의
# test_questions = [...]

# Step 3: 순차 처리 및 결과 수집
# for q in test_questions:
#     result = await degradation.invoke_with_cost(q)

# Step 4: 결과 테이블 출력
# print(f"{'호출':>4} | {'모델':>20} | {'토큰':>8} | {'비용':>10}")

# Step 5: Langfuse 이벤트 기록

print("Exercise 2: TODO — 위 코드를 완성하세요")

---
## 마무리

### 오늘 배운 것

| 가드레일 | 핵심 클래스 | 용도 |
|----------|-----------|------|
| 토큰 예산 | `BudgetTracker` | 비용 통제 (Hard + Soft Limit) |
| 시간 예산 | `TimeBudget` + `asyncio.wait_for` | 지연 방지 (Wall Clock + Per-Step) |
| 반복 제한 | `IterationLimiter` | 무한 루프 자동 종료 |
| 품질 게이트 | `quality_gate` (LLM-as-Judge) | 기준 미달 재시도/에스컬레이션 |
| 인간 개입 | LangGraph `interrupt` | 위험 작업 승인 요청 |
| 다운그레이드 | `GracefulDegradation` | Opus → Sonnet → Haiku → Refuse |
| 모니터링 | `GuardrailLogger` + Langfuse | 가드레일 트리거 추적 및 알림 |

### 핵심 원칙
1. **삼중 제어**: 예산 + 시간 + 품질을 동시에 적용 (하나만으로는 부족)
2. **방어적 기본값**: 가드레일은 기본 ON, 해제는 명시적으로
3. **외부 제어**: 가드레일은 에이전트 외부에서 불변으로 적용
4. **모니터링 필수**: 가드레일이 얼마나 자주 트리거되는지 추적

### 다음 에피소드
**EP20**: 프로덕션 에이전트 복원력 — 장애 복구와 자동 롤백

> 전체 코드는 GitHub 레포에서, 심화 내용은 커뮤니티에서